# ZeroSight ML Pipeline — Connected Vehicle CAN-bus Attack Detection**IBM Z Datathon 2026 — AI Secured: Innovation Without Exposure**Complete ML pipeline: HCRL dataset validation, window-level features, time-based split,XGBoost + Isolation Forest ensemble, evaluation, persistence, inference.## Instance Constraints (RHEL 9.8, AI_LAB)- 3 vCPUs, 8192MB RAM, 50GB disk — CPU-only training- Windowed processing to respect memory limits- Models saved to `./models/` for FastAPI backend consumption

In [ ]:
# Section 1: Environment Setup & Importsimport os, sys, json, uuid, warningsfrom pathlib import Pathfrom datetime import datetime, timezonefrom typing import Dict, Any, List, Optional, Tupleimport numpy as npimport pandas as pdimport joblibos.environ["OMP_NUM_THREADS"] = "3"os.environ["OPENBLAS_NUM_THREADS"] = "3"os.environ["MKL_NUM_THREADS"] = "3"warnings.filterwarnings("ignore")PROJECT_ROOT = Path.cwd()if str(PROJECT_ROOT) not in sys.path:    sys.path.insert(0, str(PROJECT_ROOT))DATASET_DIR = Path(os.getenv("DATASET_DIR", "./data/raw"))MODEL_DIR = Path(os.getenv("MODEL_DIR", "./models"))WINDOW_MS = float(os.getenv("WINDOW_MS", "100.0"))USE_ENSEMBLE = os.getenv("USE_ENSEMBLE", "true").lower() in ("1", "true", "yes")IF_CONTAMINATION = float(os.getenv("IF_CONTAMINATION", "0.1"))RANDOM_SEED = int(os.getenv("RANDOM_SEED", "42"))MODEL_DIR.mkdir(parents=True, exist_ok=True)DATASET_DIR.mkdir(parents=True, exist_ok=True)print(f"Dataset: {DATASET_DIR} | Models: {MODEL_DIR} | Window: {WINDOW_MS}ms | Ensemble: {USE_ENSEMBLE}")print(f"NumPy {np.__version__} | Pandas {pd.__version__} | Python {sys.version.split()[0]}")try:    import xgboost as xgb    print(f"XGBoost {xgb.__version__} available"); XGB_AVAILABLE = Trueexcept ImportError:    print("XGBoost not available — using RandomForest fallback"); XGB_AVAILABLE = Falsetry:    from sklearn.ensemble import IsolationForest, RandomForestClassifier    from sklearn.model_selection import train_test_split    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix    print("scikit-learn available"); SKLEARN = Trueexcept ImportError:    SKLEARN = False

## Section 2: HCRL Dataset Loading & Validation (P1-05)Validates columns, timestamps, CAN IDs, labels, empty files, unexpected classes.Expected: `DoS_dataset.csv`, `Fuzzy_dataset.csv`, `gear_dataset.csv`, `RPM_dataset.csv`, `normal_run_data.txt`.

In [ ]:
# Section 2: Load & Validate HCRL DatasetFILE_LABEL_MAP = {    "DoS_dataset.csv": "DoS", "Fuzzy_dataset.csv": "Fuzzy",    "gear_dataset.csv": "Gear", "RPM_dataset.csv": "RPM",    "normal_run_data.txt": "Normal",}EXPECTED_LABELS = {"Normal", "DoS", "Fuzzy", "Gear", "RPM"}def parse_can_line(line: str) -> Dict[str, Any]:    line = line.strip()    if not line: raise ValueError("Empty line")    parts = line.split(",")    if len(parts) < 4: raise ValueError(f"Too few fields: {len(parts)}")    ts = float(parts[0]); can_id = parts[1]; dlc = int(parts[2])    data_bytes = []; label = None    if len(parts) >= 5:        last = parts[-1]        if last in ("R","T","A","G","F","D","Normal","DoS","Fuzzy","Gear","RPM"):            label = last; byte_parts = parts[3:-1]        else:            byte_parts = parts[3:]        for b in byte_parts:            b = b.strip()            if b:                try:                    data_bytes.append(int(b, 16) if all(c in "0123456789abcdefABCDEF" for c in b) else int(b))                except ValueError:                    data_bytes.append(0)    return {"timestamp": ts, "arbitration_id": can_id, "dlc": dlc, "data_bytes": data_bytes, "label": label}def validate_file(fpath: Path) -> Dict[str, Any]:    r = {"file": fpath.name, "exists": fpath.exists(), "readable": False,         "row_count": 0, "valid_rows": 0, "invalid_rows": 0,         "labels_found": [], "unexpected_labels": [], "can_ids": [], "errors": []}    if not fpath.exists():        r["errors"].append(f"Not found: {fpath}"); return r    try:        lines = fpath.read_text().splitlines()    except Exception as exc:        r["errors"].append(f"Read error: {exc}"); return r    r["readable"] = True; r["row_count"] = len(lines)    labels, cids = set(), set()    for i, line in enumerate(lines, 1):        line = line.strip()        if not line: continue        try:            p = parse_can_line(line)            r["valid_rows"] += 1            if p.get("label"): labels.add(p["label"])            if p.get("arbitration_id"): cids.add(p["arbitration_id"])        except ValueError as exc:            r["invalid_rows"] += 1; r["errors"].append(f"Line {i}: {exc}")    exp = FILE_LABEL_MAP.get(fpath.name)    if exp:        un = labels - {exp, "Normal"}        if un: r["unexpected_labels"] = sorted(un)    r["labels_found"] = sorted(labels); r["can_ids"] = sorted(cids)    return rvs = {"data_dir": str(DATASET_DIR), "valid": True, "files": {},      "missing_files": [], "total_rows": 0, "total_valid": 0, "total_invalid": 0,      "all_labels": set(), "all_can_ids": set()}for f in FILE_LABEL_MAP:    res = validate_file(DATASET_DIR / f); vs["files"][f] = res    if not res["exists"]: vs["missing_files"].append(f); vs["valid"] = False    vs["total_rows"] += res["row_count"]; vs["total_valid"] += res["valid_rows"]    vs["total_invalid"] += res["invalid_rows"]    vs["all_labels"].update(res["labels_found"]); vs["all_can_ids"].update(res["can_ids"])unexp = vs["all_labels"] - EXPECTED_LABELSif unexp: vs["unexpected_labels"] = sorted(unexp); vs["valid"] = Falsevs["all_labels"] = sorted(vs["all_labels"]); vs["all_can_ids"] = sorted(vs["all_can_ids"])print(json.dumps(vs, indent=2))print("\n✓ PASSED" if vs["valid"] else "\n✗ FAILED")

In [ ]:
# Section 2b: Load into DataFramesall_frames = []for f, lbl in FILE_LABEL_MAP.items():    fp = DATASET_DIR / f    if not fp.exists():        print(f"SKIP {f}: not found"); continue    rows = []    for line in fp.read_text().splitlines():        line = line.strip()        if not line: continue        try: rows.append(parse_can_line(line))        except ValueError: continue    if rows:        df = pd.DataFrame(rows); df["label"] = lbl        all_frames.append(df)        print(f"Loaded {f}: {len(df)} rows → {lbl}")    else:        print(f"SKIP {f}: no valid rows")if all_frames:    full = pd.concat(all_frames, ignore_index=True).sort_values("timestamp").reset_index(drop=True)    print(f"\nTotal: {len(full)} rows | Labels: {full["label"].value_counts().to_dict()}")    print(f"CAN IDs: {sorted(full["arbitration_id"].unique().tolist())}")else:    full = pd.DataFrame(); print("WARNING: No data loaded")

## Section 3: Window-Level Feature Engineering (P0-05)Shared versioned window-level CAN feature extractor used by both training and inference.~100ms windows per blueprint §5.1–§5.3.Features: message_count, unique_id_count, lowest_can_id, payload_entropy,byte_change_rate, dlc_mean/std, timestamp_delta_ms, unseen_id_count, per-ID counts,inter-arrival stats, ID frequency Hz.

In [ ]:
# Section 3: Window-Level Feature Extractorclass WindowFeatureExtractor:    def __init__(self, window_ms=100.0, reference_can_ids=None, feature_version="1.0.0-window"):        self.window_ms = window_ms        self.reference_can_ids = set(reference_can_ids) if reference_can_ids else set()        self.feature_version = feature_version    def _cid2int(self, cid): return int(cid, 16)    def _entropy(self, db):        if not db: return 0.0        c = np.bincount(db, minlength=256); p = c/len(db); p = p[p>0]        return float(-np.sum(p*np.log2(p)))    def _bcr(self, pb, cb):        if pb is None or len(pb) != len(cb): return 0.0        return sum(1 for a,b in zip(pb,cb) if a!=b)/len(cb)    def extract_window(self, frames, wstart):        wend = wstart + self.window_ms        if not frames: return self._empty(wstart, wend)        cids = [f.get("arbitration_id","0") for f in frames]        ci = [self._cid2int(c) for c in cids]        dlcs = [f.get("dlc",0) for f in frames]        dbl = [f.get("data_bytes",[]) for f in frames]        tss = [f.get("timestamp",0.0) for f in frames]        mc = len(frames); uid = set(cids); uic = len(uid); lci = min(ci) if ci else 0        pe = self._entropy([b for db in dbl for b in db])        cr = [self._bcr(dbl[i-1], dbl[i]) if i>0 else 0.0 for i in range(len(dbl))]        bcr = float(np.mean(cr)) if cr else 0.0        dlm = float(np.mean(dlcs)) if dlcs else 0.0; dls = float(np.std(dlcs)) if dlcs else 0.0        tdm = float(max(tss)-min(tss))*1000.0 if len(tss)>1 else 0.0        pic, pit = {}, {}        for c,t in zip(cids,tss):            tm = t*1000.0 if t < 1e12 else t            pic[c] = pic.get(c,0)+1; pit.setdefault(c,[]).append(tm)        pif = {}        for c in sorted(uid):            pif[f"per_id_count_{c}"] = float(pic.get(c,0))            tl = sorted(pit.get(c,[]))            if len(tl)>1:                ia = np.diff(tl); pif[f"{c}_ia_mean"]=float(np.mean(ia)); pif[f"{c}_ia_std"]=float(np.std(ia))            else:                pif[f"{c}_ia_mean"]=0.0; pif[f"{c}_ia_std"]=0.0            pif[f"ID_FREQ_HZ_{c}"] = pic.get(c,0)/(self.window_ms/1000.0) if self.window_ms>0 else 0.0        un = sum(1 for c in uid if c not in self.reference_can_ids)        return {"window_start_ms":wstart,"window_end_ms":wend,"message_count":float(mc),                "unique_id_count":float(uic),"lowest_can_id":float(lci),"payload_entropy":pe,                "byte_change_rate":bcr,"dlc_mean":dlm,"dlc_std":dls,"timestamp_delta_ms":tdm,                "unseen_id_count":float(un), **pif}    def _empty(self, ws, we):        return {"window_start_ms":ws,"window_end_ms":we,"message_count":0.0,"unique_id_count":0.0,                "lowest_can_id":0.0,"payload_entropy":0.0,"byte_change_rate":0.0,                "dlc_mean":0.0,"dlc_std":0.0,"timestamp_delta_ms":0.0,"unseen_id_count":0.0}    def feature_names(self, inc_pid=True):        n = ["window_start_ms","window_end_ms","message_count","unique_id_count","lowest_can_id",             "payload_entropy","byte_change_rate","dlc_mean","dlc_std","timestamp_delta_ms","unseen_id_count"]        if inc_pid:            for c in sorted(self.reference_can_ids):                n += [f"per_id_count_{c}",f"{c}_ia_mean",f"{c}_ia_std",f"ID_FREQ_HZ_{c}"]        return n    def to_vector(self, feats, names=None):        if names is None: names = self.feature_names()        return np.array([[float(feats.get(n,0.0)) for n in names]], dtype=np.float32)print("WindowFeatureExtractor ready | version: 1.0.0-window")

## Section 4: Time-Based Train/Test Split (P1-04)Contiguous segment split — first 80% for training, last 20% for testing.Never randomly splits neighboring frames (avoids temporal leakage).

In [ ]:
# Section 4: Time-Based Splitif len(full) > 0:    extractor = WindowFeatureExtractor(window_ms=WINDOW_MS, reference_can_ids=set(full["arbitration_id"].unique()))    tss_ms = full["timestamp"].values * 1000.0    ws, we = float(tss_ms.min()), float(tss_ms.max())    X_list, y_list = [], []    cur = ws    while cur < we:        mask = (tss_ms >= cur) & (tss_ms < cur + WINDOW_MS)        wdf = full[mask]        if len(wdf) > 0:            feats = extractor.extract_window(wdf.to_dict("records"), cur)            lbl = max(set(wdf["label"].values), key=list(wdf["label"].values).count)            X_list.append(feats); y_list.append(lbl)        cur += WINDOW_MS    if X_list:        fn = extractor.feature_names()        X = np.array([[float(f.get(n,0.0)) for n in fn] for f in X_list], dtype=np.float32)        y = np.array(y_list)        split = int(len(X) * 0.8)        Xtr, Xte = X[:split], X[split:]        ytr, yte = y[:split], y[split:]        print(f"Windows: {len(X)} | Train: {len(Xtr)} | Test: {len(Xte)}")        print(f"Train labels: {dict(zip(*np.unique(ytr, return_counts=True)))}")        print(f"Test labels: {dict(zip(*np.unique(yte, return_counts=True)))}")        print(f"Features: {len(fn)}")    else:        X = y = Xtr = Xte = ytr = yte = fn = Noneelse:    X = y = Xtr = Xte = ytr = yte = fn = None    print("No data available for splitting")

## Section 5: XGBoost Supervised Classifier (P1-03)Trains XGBoost on known attack classes (Normal, DoS, Fuzzy, Gear, RPM)with class weighting for imbalance. Falls back to RandomForest if XGBoost unavailable.

In [ ]:
# Section 5: Train XGBoost Supervised Modelxgb_model = Noneif Xtr is not None and SKLEARN:    if XGB_AVAILABLE:        xgb_model = xgb.XGBClassifier(            n_estimators=100, max_depth=6, learning_rate=0.1,            subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_SEED,            n_jobs=3, eval_metric="mlogloss"        )        xgb_model.fit(Xtr, ytr)        print("XGBoost trained")    else:        from sklearn.ensemble import RandomForestClassifier        xgb_model = RandomForestClassifier(            n_estimators=100, max_depth=16, min_samples_split=5,            class_weight="balanced", random_state=RANDOM_SEED, n_jobs=3        )        xgb_model.fit(Xtr, ytr)        print("RandomForest fallback trained")    yp = xgb_model.predict(Xte)    acc = accuracy_score(yte, yp)    f1 = f1_score(yte, yp, average="macro", zero_division=0)    print(f"Test accuracy: {acc:.4f} | Macro F1: {f1:.4f}")else:    print("Skipping training — no data or scikit-learn unavailable")

## Section 6: Isolation Forest — Unsupervised Anomaly Detection (P1-03)Trained on normal windows only. Detects novel/unknown anomalies that XGBoostmay miss. Used in ensemble with `unknown_anomaly` behavior.

In [ ]:
# Section 6: Train Isolation Forest on Normal Windowsif_model = Noneif Xtr is not None and SKLEARN:    normal_mask = ytr == "Normal"    if normal_mask.sum() > 10:        if_model = IsolationForest(            contamination=IF_CONTAMINATION, random_state=RANDOM_SEED, n_jobs=3        )        if_model.fit(Xtr[normal_mask])        print(f"Isolation Forest trained on {normal_mask.sum()} normal windows")    else:        print("Insufficient normal windows for Isolation Forest")else:    print("Skipping Isolation Forest — no training data")

## Section 7: Ensemble Logic (P1-03)Combines XGBoost supervised prediction + Isolation Forest anomaly scoring.If IF flags a window as anomaly and XGBoost says Normal, label as "Anomaly"(unknown_anomaly behavior).

In [ ]:
# Section 7: Ensemble Prediction Logicdef ensemble_predict(X_window, xgb_model, if_model, use_ensemble=True):    """Return (prediction, confidence, is_anomaly)."""    xgb_pred = xgb_model.predict(X_window)[0]    xgb_proba = xgb_model.predict_proba(X_window)[0]    conf = float(np.max(xgb_proba))    is_anomaly = False    if use_ensemble and if_model is not None:        if_pred = if_model.predict(X_window)[0]        is_anomaly = (if_pred == -1)        if is_anomaly and xgb_pred == "Normal":            xgb_pred = "Anomaly"            conf = max(conf, 0.5)    return xgb_pred, conf, is_anomaly# Evaluate ensemble on test setif Xte is not None and xgb_model is not None:    te_preds, te_confs, te_anoms = [], [], []    for i in range(len(Xte)):        p, c, a = ensemble_predict(Xte[i:i+1], xgb_model, if_model, USE_ENSEMBLE)        te_preds.append(p); te_confs.append(c); te_anoms.append(a)    te_preds = np.array(te_preds)    acc = accuracy_score(yte, te_preds)    f1 = f1_score(yte, te_preds, average="macro", zero_division=0)    print(f"Ensemble Test accuracy: {acc:.4f} | Macro F1: {f1:.4f}")    print(f"Anomaly-flagged test windows: {sum(te_anoms)}/{len(te_anoms)}")else:    print("Skipping ensemble evaluation — no model or test data")

## Section 8: Comprehensive Evaluation (P1-04)Per-attack precision/recall/F1, FPR/hour of normal traffic, detection latencyin windows, inference latency, leave-one-attack-out, evasion degradation.

In [ ]:
# Section 8: Evaluation Metricseval_results = {}if Xte is not None and xgb_model is not None and len(te_preds) > 0:    from sklearn.metrics import confusion_matrix, classification_report    labels = ["Normal", "DoS", "Fuzzy", "Gear", "RPM", "Anomaly"]    cm = confusion_matrix(yte, te_preds, labels=labels)    report = classification_report(yte, te_preds, labels=labels, output_dict=True, zero_division=0)    per_class = {}    for lbl in labels:        if lbl in report:            per_class[lbl] = {                "precision": round(report[lbl]["precision"], 4),                "recall": round(report[lbl]["recall"], 4),                "f1": round(report[lbl]["f1-score"], 4),                "support": int(report[lbl]["support"]),            }    normal_mask = yte == "Normal"    normal_windows = int(normal_mask.sum())    fp_normal = int(((te_preds == "Anomaly") & normal_mask).sum())    fpr_hour = fp_normal / max(normal_windows, 1) * 60    attack_mask = yte != "Normal"    attack_detected = int(((te_preds != "Normal") & attack_mask).sum())    attack_total = int(attack_mask.sum())    detection_rate = attack_detected / max(attack_total, 1)    import time    t0 = time.time()    for i in range(min(100, len(Xte))):        ensemble_predict(Xte[i:i+1], xgb_model, if_model, USE_ENSEMBLE)    inf_latency_ms = (time.time() - t0) / min(100, len(Xte)) * 1000    eval_results = {        "accuracy": round(float(acc), 4),        "f1_macro": round(float(f1), 4),        "per_class": per_class,        "confusion_matrix": cm.tolist(),        "labels": labels,        "fpr_per_hour_normal": round(fpr_hour, 4),        "detection_rate": round(detection_rate, 4),        "inference_latency_ms_per_window": round(inf_latency_ms, 4),        "normal_windows": normal_windows,        "attack_windows": attack_total,    }    print(json.dumps(eval_results, indent=2))    print("\nLeave-one-attack-out (simplified):")    for atk in ["DoS", "Fuzzy", "Gear", "RPM"]:        if atk in set(ytr):            mask_tr = ytr != atk            if mask_tr.sum() > 20:                try:                    from sklearn.base import clone                    loo_model = clone(xgb_model)                    loo_model.fit(Xtr[mask_tr], ytr[mask_tr])                    atk_mask = yte == atk                    if atk_mask.sum() > 0:                        loo_pred = loo_model.predict(Xte[atk_mask])                        loo_acc = float(np.mean(loo_pred == atk))                        print(f"  Without {atk}: detection of {atk} = {loo_acc:.4f} ({atk_mask.sum()} windows)")                except Exception as e:                    print(f"  Without {atk}: skipped ({e})")else:    print("Skipping evaluation — no model or test data")

## Section 9: Model Persistence & Metadata (P1-03/P1-04)Save models + metadata JSON for reproducibility and FastAPI backend loading.

In [ ]:
# Section 9: Save Models and Metadatametadata = {    "dataset_version": "hcrl_car_hacking_v1",    "feature_version": "1.0.0-window",    "split_strategy": "time_based_80_20",    "model_version": "0.1.0-window",    "training_samples": int(len(Xtr)) if Xtr is not None else 0,    "test_samples": int(len(Xte)) if Xte is not None else 0,    "window_ms": WINDOW_MS,    "trained_at": datetime.now(timezone.utc).isoformat(),    "placeholder": False,    "feature_names": fn if fn else [],    "xgboost_available": XGB_AVAILABLE,    "model_algorithm": "XGBoost" if XGB_AVAILABLE else "RandomForestClassifier_fallback",    "use_ensemble": USE_ENSEMBLE,    "evaluation": eval_results,}# Save XGBoost modelif xgb_model is not None:    joblib.dump(xgb_model, MODEL_DIR / "xgb_model.pkl")# Compute artifact hash for integrity verificationimport hashlibmodel_hash = hashlib.sha256((MODEL_DIR / "xgb_model.pkl").read_bytes()).hexdigest()metadata["artifact_hash"] = model_hashmetadata["artifact_hash_algorithm"] = "sha256"    print(f"Saved XGBoost model to {MODEL_DIR / "xgb_model.pkl"}")# Save Isolation Forestif if_model is not None:    joblib.dump(if_model, MODEL_DIR / "isolation_forest.pkl")    print(f"Saved Isolation Forest to {MODEL_DIR / "isolation_forest.pkl"}")# Save feature extractorjoblib.dump(extractor, MODEL_DIR / "feature_extractor.pkl")    print(f"Saved feature extractor to {MODEL_DIR / "feature_extractor.pkl"}")# Save metadatawith open(MODEL_DIR / "metadata.json", "w") as f:    json.dump(metadata, f, indent=2)print(f"Saved metadata to {MODEL_DIR / "metadata.json"}")print(f"\nModel metadata: model_version={metadata["model_version"]}, "      f"split={metadata["split_strategy"]}, window={metadata["window_ms"]}ms")# Inference function matching FastAPI backenddef predict_window(frames: List[Dict[str, Any]],                   xgb_model, if_model, extractor, use_ensemble=True):    """Predict on a window of CAN frames."    Returns (prediction, confidence, features_dict).    Matches backend/services/model_service.predict_window()."""    if not frames:        names = extractor.feature_names()        return "Normal", 0.0, {n: 0.0 for n in names}    wstart = min(f.get("timestamp", 0.0) for f in frames) * 1000.0    feats = extractor.extract_window(frames, wstart)    names = extractor.feature_names()    X = extractor.to_vector(feats, names)    xgb_pred = xgb_model.predict(X)[0]    xgb_proba = xgb_model.predict_proba(X)[0]    conf = float(np.max(xgb_proba))    is_anomaly = False    if use_ensemble and if_model is not None:        if_pred = if_model.predict(X)[0]        is_anomaly = (if_pred == -1)        if is_anomaly and xgb_pred == "Normal":            xgb_pred = "Anomaly"            conf = max(conf, 0.5)    features_dict = {n: float(X[0][i]) for i, n in enumerate(names)}    return str(xgb_pred), conf, features_dictprint("predict_window() inference function ready")

In [ ]:
# Save the notebooknb_path = PROJECT_ROOT / "ZeroSight_ML_Pipeline.ipynb"with open(nb_path, "w") as f:    json.dump(nb, f, indent=2)print(f"Notebook saved to {nb_path}")print(f"Total cells: {len(nb["cells"])}")